# 03 — Gold features and per-station baselines

Reads the conformed silver tables and writes the station-day feature mart to `lh_gold`.

**Outputs**

| Table | Grain | Purpose |
|---|---|---|
| `gold_entity_baseline` | station × metric | robust normal (median/MAD), weekday profile, scale |
| `gold_entity_day_features` | station × date | one wide row per station-day with levels, deviations, lags, rolling/EWMA/CUSUM |
| `gold_deviation_daily` | station × date × metric | long-form, explainable deviations |
| `gold_data_quality` | metric/value | build checks |

**Design rules**

1. **Complete grid.** `5 stations × 90 dates = 450` rows. `silver_station_day` only has
   observed days, so the grid is built explicitly and `is_observed` marks the rest.
   **Missing is not zero** — unobserved levels stay null and are never treated as a loss.
2. **No leakage.** Baselines are estimated on the first **28 calendar days** only and then
   applied forward. Rows inside that window are labelled `baseline_phase = estimation`;
   everything after is `scored`. Sequential features use a prior-day shift.
3. **Robust.** Median/MAD with an IQR/std floor, so a static or sparse seed signal cannot
   produce a divide-by-zero.
4. **Explainable.** Every deviation carries `expected`, `deviation`, `robust_z`, direction
   and a plain-language reason. Risk rules in `nb_04` consume these columns directly.


In [ ]:
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

# --- sandbox identifiers ---------------------------------------------------
WORKSPACE_ID = "355cd573-8b17-430a-b606-c6b9424d36a1"
SILVER_ID = "b6d8fe8b-bad2-4ac5-a605-f0b13893df7f"   # lh_silver
GOLD_ID = "df9a08e5-b370-4c2a-b608-6d0f326003ef"     # lh_gold

SILVER = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_ID}"
GOLD = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{GOLD_ID}"

BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
COMPUTED_AT = datetime.now(timezone.utc).replace(tzinfo=None)

# --- baseline / deviation configuration ------------------------------------
BASELINE_DAYS = 28        # first N calendar days used to learn "normal"
MIN_BASELINE_OBS = 14     # warn below this (do not fail)
MAD_TO_SIGMA = 1.4826
IQR_TO_SIGMA = 1.349
OUTLIER_Z = 3.0
DEVIATED_Z = 2.0
FLAT_Z = 0.5
CUSUM_K = 0.5
Z_CLIP = 25.0             # cap pathological z values from static seed signals
WEEKDAY_SHRINK = 0.75     # 75% weekday median + 25% overall median

METRICS = [
    # name, source column, group, higher_is_bad, relative floor, absolute floor, label
    ("oil",            "actual_oil",      "production", False, 0.01,  25.0,  "oil"),
    ("deferred_oil",   "deferred_oil",    "loss",       True,  0.01,  25.0,  "deferred oil"),
    ("deferment_pct",  "deferment_pct",   "loss",       True,  0.001, 0.01,  "deferment %"),
    ("target_gap_pct", "target_gap_pct",  "production", True,  0.001, 0.02,  "gap to target"),
    ("uptime",         "uptime_avg",      "process",    False, 0.01,  0.5,   "uptime"),
    ("thp",            "thp_avg",         "process",    False, 0.001, 5.0,   "THP"),
    ("bean",           "bean_avg",        "process",    False, 0.001, 1.0,   "bean/choke"),
    ("water_cut",      "water_rf_avg",    "process",    True,  0.001, 0.01,  "water cut"),
]
METRIC_NAMES = [m[0] for m in METRICS]
METRIC_META = {
    m[0]: {"column": m[1], "group": m[2], "higher_is_bad": m[3],
           "floor_ratio": m[4], "min_scale": m[5], "label": m[6]}
    for m in METRICS
}
CORE_METRICS = ["oil", "uptime", "thp", "bean", "water_cut", "deferment_pct"]


def read_silver(name: str):
    return spark.read.format("delta").load(f"{SILVER}/Tables/silver_{name}")


def write_gold(name: str, sdf) -> None:
    """Overwrite the Delta path, then register it in the gold lakehouse."""
    path = f"{GOLD}/Tables/{name}"
    (sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path))
    spark.sql(f"CREATE TABLE IF NOT EXISTS {name} USING DELTA LOCATION '{path}'")
    print(f"{name}: {sdf.count():,} rows")


print(f"Batch: {BATCH_ID}")

In [ ]:
# --- read silver and build the complete station-date grid ------------------
stations = (
    read_silver("dim_station")
    .select(
        F.col("flowstation_id").alias("station_id"),
        "station_name",
        "asset",
    )
)

dates = (
    read_silver("dim_date")
    .select("date", "weekday", "is_weekend")
)

sd = (
    read_silver("station_day")
    .select(
        "station_id", "date",
        "actual_oil", "target_oil", "actual_gas", "actual_water", "actual_gross",
        "uptime_avg", "thp_avg", "bean_avg", "water_rf_avg",
        "wells_total", "wells_producing",
        "deferred_oil", "event_count", "coded_event_count",
        "potential_oil", "deferment_pct", "target_gap", "target_gap_pct",
    )
    .withColumn("is_observed", F.lit(True))
)

prices = read_silver("oil_prices").select("date", "oil_price_usd")
owners = read_silver("owner_map").select(
    F.col("station_id"), "owner_name", "owner_upn"
)

grid = (
    stations.crossJoin(dates)
    .join(sd, ["station_id", "date"], "left")
    .join(prices, "date", "left")
    .join(owners, "station_id", "left")
    .withColumn("is_observed", F.coalesce(F.col("is_observed"), F.lit(False)))
    # Spark dayofweek: Sunday=1 ... Saturday=7. Convert to Monday=0 ... Sunday=6.
    .withColumn("weekday_num", ((F.dayofweek("date") + 5) % 7).cast("int"))
)

grid_pd = grid.toPandas()
grid_pd["date"] = pd.to_datetime(grid_pd["date"])
grid_pd = grid_pd.sort_values(["station_id", "date"]).reset_index(drop=True)
for c in ["is_observed", "is_weekend"]:
    grid_pd[c] = grid_pd[c].fillna(False).astype(bool)

print(f"grid rows        : {len(grid_pd):,}")
print(f"stations         : {grid_pd['station_id'].nunique()}")
print(f"dates            : {grid_pd['date'].nunique()}")
print(f"observed station-days: {int(grid_pd['is_observed'].sum()):,}")
print(f"unobserved            : {int((~grid_pd['is_observed']).sum()):,}")
display(grid_pd.groupby("station_name", dropna=False).agg(
    dates=("date", "size"),
    observed=("is_observed", "sum"),
))

In [ ]:
# --- 1. per-station robust baselines ---------------------------------------
# Estimation window = first BASELINE_DAYS calendar dates. It is applied forward
# to every date; rows inside it are labelled later as `estimation`.
date_values = np.sort(grid_pd["date"].unique())
EST_START = pd.Timestamp(date_values[0])
EST_END = pd.Timestamp(date_values[min(BASELINE_DAYS, len(date_values)) - 1])
BASELINE_VERSION = f"gold-v1-{EST_END:%Y%m%d}"


def robust_scale(vals: pd.Series, floor_ratio: float, min_scale: float) -> float:
    """MAD-based sigma with IQR/std, relative and absolute floors (never zero)."""
    med = float(vals.median())
    mad = float((vals - med).abs().median())
    q25, q75 = float(vals.quantile(0.25)), float(vals.quantile(0.75))
    std = float(vals.std(ddof=0)) if len(vals) > 1 else 0.0
    # Robust scale first. Std is deliberately NOT used as the scale: one
    # outlier would inflate it and hide the very deviations we look for.
    return max(
        MAD_TO_SIGMA * mad,
        (q75 - q25) / IQR_TO_SIGMA,
        abs(med) * floor_ratio,
        min_scale,
    )


baseline_rows = []
for sid, sdf in grid_pd[grid_pd.is_observed].groupby("station_id"):
    est = sdf[(sdf["date"] >= EST_START) & (sdf["date"] <= EST_END)]
    station_name = sdf["station_name"].dropna().iloc[0]
    for name, meta in METRIC_META.items():
        col = meta["column"]
        usable = est[est[col].notna()]
        vals = usable[col]
        if len(vals) < MIN_BASELINE_OBS:
            print(f"  warn: {station_name} / {name}: only {len(vals)} baseline observations")

        med = float(vals.median()) if len(vals) else np.nan
        scale = robust_scale(vals, meta["floor_ratio"], meta["min_scale"]) if len(vals) else np.nan
        weekday_expected = {}
        for wd in range(7):
            wv = usable.loc[usable["weekday_num"] == wd, col]
            wm = float(wv.median()) if len(wv) else np.nan
            weekday_expected[wd] = (
                WEEKDAY_SHRINK * wm + (1 - WEEKDAY_SHRINK) * med
                if pd.notna(wm) and pd.notna(med) else med
            )

        finite_wd = [v for v in weekday_expected.values() if pd.notna(v)]
        spread = (
            max(finite_wd) / min(finite_wd)
            if finite_wd and min(finite_wd) > 0 else np.nan
        )

        row = {
            "entity_type": "station",
            "entity_id": int(sid),
            "entity_name": station_name,
            "metric": name,
            "metric_group": meta["group"],
            "higher_is_bad": bool(meta["higher_is_bad"]),
            "baseline_version": BASELINE_VERSION,
            "estimation_start": EST_START.date(),
            "estimation_end": EST_END.date(),
            "n_obs": int(len(vals)),
            "median": med,
            "mad": float((vals - med).abs().median()) if len(vals) else np.nan,
            "robust_scale": scale,
            "mean": float(vals.mean()) if len(vals) else np.nan,
            "stddev": float(vals.std(ddof=0)) if len(vals) > 1 else np.nan,
            "p10": float(vals.quantile(0.10)) if len(vals) else np.nan,
            "p90": float(vals.quantile(0.90)) if len(vals) else np.nan,
            "min_value": float(vals.min()) if len(vals) else np.nan,
            "max_value": float(vals.max()) if len(vals) else np.nan,
            "seasonal_spread": spread,
            "computed_at_utc": COMPUTED_AT,
        }
        for wd in range(7):
            row[f"weekday_expected_{wd}"] = weekday_expected[wd]
        baseline_rows.append(row)

baseline_df = pd.DataFrame(baseline_rows)
print(f"baselines: {len(baseline_df):,} rows "
      f"({baseline_df['entity_id'].nunique()} stations x {len(METRIC_META)} metrics)")
print(f"estimation window: {EST_START.date()} -> {EST_END.date()}  ({BASELINE_VERSION})")
display(
    baseline_df.pivot(index="entity_name", columns="metric", values="median")
    .round(3)
)

In [ ]:
# --- 2. daily deviations + sequential features -----------------------------
# All maths is done on observed rows; the full calendar is kept so that
# lags/rolling windows respect missing days instead of silently compressing them.
BASE = baseline_df.set_index(["entity_id", "metric"])


def slope_7(y) -> float:
    y = np.asarray(y, dtype=float)
    mask = ~np.isnan(y)
    if mask.sum() < 4:
        return np.nan
    return float(np.polyfit(np.arange(len(y))[mask], y[mask], 1)[0])


def add_cusum(z: pd.Series, observed: pd.Series, k: float = CUSUM_K):
    """Two-sided CUSUM in robust-z units; state survives missing days."""
    pos, neg = [], []
    cp = cn = 0.0
    for zv, obs in zip(z.to_numpy(), observed.to_numpy()):
        if obs and pd.notna(zv):
            cp = max(0.0, cp + zv - k)
            cn = max(0.0, cn - zv - k)
        pos.append(cp if obs else np.nan)
        neg.append(cn if obs else np.nan)
    return pos, neg


RATIO_METRICS = {"deferment_pct", "water_cut", "target_gap_pct"}


def reason_text(row, name: str) -> str:
    z = row[f"z_{name}"]
    direction = row[f"dir_{name}"]
    if pd.isna(z) or direction in ("", "flat"):
        return ""
    expected = row[f"expected_{name}"]
    value = row[METRIC_META[name]["column"]]
    change = ""
    if pd.notna(expected) and pd.notna(value):
        if name in RATIO_METRICS and abs(value - expected) >= 0.0005:
            change = f"{abs(value - expected) * 100:.0f}pp "
        elif abs(expected) > 1e-12:
            change = f"{abs(value - expected) / abs(expected) * 100:.0f}% "
    return f"{METRIC_META[name]['label']} {change}{direction} its normal (z={z:+.1f})"


feature_frames = []
for sid, sdf in grid_pd.groupby("station_id"):
    s = sdf.sort_values("date").copy().reset_index(drop=True)
    s["baseline_phase"] = np.where(s["date"] <= EST_END, "estimation", "scored")

    for name, meta in METRIC_META.items():
        col = meta["column"]
        b = BASE.loc[(sid, name)]
        med, scale = b["median"], b["robust_scale"]
        s[f"expected_{name}"] = [
            b[f"weekday_expected_{int(wd)}"] if pd.notna(b[f"weekday_expected_{int(wd)}"]) else med
            for wd in s["weekday_num"]
        ]
        s[f"dev_{name}"] = s[col] - s[f"expected_{name}"]
        s[f"z_{name}"] = (s[f"dev_{name}"] / scale).clip(-Z_CLIP, Z_CLIP) if pd.notna(scale) and scale > 0 else np.nan
        s[f"pct_dev_{name}"] = s[f"dev_{name}"] / s[f"expected_{name}"].abs().replace(0, np.nan)
        s[f"dir_{name}"] = np.select(
            [s[f"z_{name}"].isna(), s[f"z_{name}"].abs() < FLAT_Z, s[f"z_{name}"] > 0],
            ["", "flat", "above"],
            default="below",
        )
        s[f"outlier_{name}"] = s[f"z_{name}"].abs() >= OUTLIER_Z
        s[f"missing_{name}"] = s[col].isna() & s["is_observed"]

    for name in CORE_METRICS:
        col = METRIC_META[name]["column"]
        v = s[col]
        s[f"{name}_lag_1"] = v.shift(1)
        s[f"{name}_lag_7"] = v.shift(7)
        s[f"{name}_delta_1"] = v - s[f"{name}_lag_1"]
        s[f"{name}_delta_7"] = v - s[f"{name}_lag_7"]
        s[f"{name}_roll_mean_7"] = v.shift(1).rolling(7, min_periods=3).mean()
        s[f"{name}_roll_std_7"] = v.shift(1).rolling(7, min_periods=3).std(ddof=0)
        s[f"{name}_roll_mean_28"] = v.shift(1).rolling(28, min_periods=7).mean()
        s[f"{name}_ewma_7_prev"] = (
            v.ewm(span=7, adjust=False, min_periods=3).mean().shift(1)
        )
        s[f"{name}_slope_7"] = v.rolling(7, min_periods=4).apply(slope_7, raw=True)
        cpos, cneg = add_cusum(s[f"z_{name}"], s["is_observed"])
        s[f"{name}_cusum_pos"] = cpos
        s[f"{name}_cusum_neg"] = cneg

    # --- composite explainability ------------------------------------------
    zcols = [f"z_{n}" for n in METRIC_NAMES]
    absz = s[zcols].abs()
    s["deviation_score"] = absz.max(axis=1)
    s["signals_deviated"] = (absz >= DEVIATED_Z).sum(axis=1).astype(int)

    def top_signal(row):
        pairs = [(row[f"z_{n}"], n) for n in METRIC_NAMES if pd.notna(row[f"z_{n}"])]
        if not pairs:
            return "", np.nan, ""
        z, n = max(pairs, key=lambda t: abs(t[0]))
        return n, z, row[f"dir_{n}"]

    tops = s.apply(top_signal, axis=1, result_type="expand")
    s["top_signal"] = tops[0]
    s["top_signal_z"] = tops[1]
    s["top_signal_direction"] = tops[2]

    def signals_json(row):
        items = []
        for n in METRIC_NAMES:
            z = row[f"z_{n}"]
            if pd.isna(z) or row[f"dir_{n}"] in ("", "flat"):
                continue
            items.append({
                "metric": n,
                "label": METRIC_META[n]["label"],
                "z": round(float(z), 2),
                "direction": row[f"dir_{n}"],
                "value": None if pd.isna(row[METRIC_META[n]["column"]]) else float(row[METRIC_META[n]["column"]]),
                "expected": None if pd.isna(row[f"expected_{n}"]) else float(row[f"expected_{n}"]),
            })
        items.sort(key=lambda x: abs(x["z"]), reverse=True)
        return json.dumps(items[:5])

    def reasons_text(row):
        parts = []
        for n in METRIC_NAMES:
            z = row[f"z_{n}"]
            if pd.notna(z) and abs(z) >= 1.5:
                txt = reason_text(row, n)
                if txt:
                    parts.append((abs(z), txt))
        parts.sort(reverse=True, key=lambda t: t[0])
        return "; ".join(t for _, t in parts[:3])

    s["signals_json"] = s.apply(signals_json, axis=1)
    s["top_reasons"] = s.apply(reasons_text, axis=1)

    # --- value context (barrels / money) -----------------------------------
    price = s["oil_price_usd"]
    s["deferred_value_usd"] = s["deferred_oil"] * price
    s["potential_value_usd"] = s["potential_oil"] * price
    s["target_gap_value_usd"] = s["target_gap"] * price

    for c in ["deferred_oil", "deferred_value_usd"]:
        s[f"{c}_roll_mean_7"] = s[c].shift(1).rolling(7, min_periods=3).mean()
        s[f"{c}_roll_sum_7"] = s[c].shift(1).rolling(7, min_periods=3).sum()
    s["event_count_roll_sum_7"] = s["event_count"].shift(1).rolling(7, min_periods=3).sum()
    s["coded_event_count_roll_sum_7"] = (
        s["coded_event_count"].shift(1).rolling(7, min_periods=3).sum()
    )

    s["baseline_version"] = BASELINE_VERSION
    s["feature_version"] = "gold-features-v1"
    s["computed_at_utc"] = COMPUTED_AT
    feature_frames.append(s)

features_pd = pd.concat(feature_frames, ignore_index=True)

# Make string columns Spark-safe when a whole column is empty.
for c in ["top_signal", "top_signal_direction", "top_reasons", "weekday", "asset",
          "station_name", "owner_name", "owner_upn"]:
    if c in features_pd.columns:
        features_pd[c] = features_pd[c].fillna("").astype(str)

print(f"features: {len(features_pd):,} rows x {len(features_pd.columns)} cols")
display(
    features_pd.loc[features_pd.is_observed]
    .sort_values("deviation_score", ascending=False)
    .loc[:, ["station_name", "date", "actual_oil", "expected_oil", "z_oil",
             "z_uptime", "z_deferment_pct", "deviation_score", "top_signal", "top_reasons"]]
    .head(12)
)

In [ ]:
# --- 3. long-form deviation table -----------------------------------------
dev_frames = []
for name, meta in METRIC_META.items():
    col = meta["column"]
    tmp = features_pd.loc[features_pd["is_observed"]].copy()
    tmp = tmp.loc[tmp[col].notna()]
    tmp["metric"] = name
    tmp["value"] = tmp[col]
    tmp["metric_group"] = meta["group"]
    tmp["higher_is_bad"] = bool(meta["higher_is_bad"])
    dev_frames.append(tmp.loc[:, [
        "station_id", "station_name", "date", "metric", "metric_group",
        "higher_is_bad", "value", f"expected_{name}", f"dev_{name}",
        f"pct_dev_{name}", f"z_{name}", f"dir_{name}", f"outlier_{name}",
        "baseline_version", "baseline_phase",
    ]].rename(columns={
        f"expected_{name}": "expected",
        f"dev_{name}": "deviation",
        f"pct_dev_{name}": "pct_deviation",
        f"z_{name}": "robust_z",
        f"dir_{name}": "direction",
        f"outlier_{name}": "is_outlier",
    }))

deviations_pd = pd.concat(dev_frames, ignore_index=True)
deviations_pd["entity_type"] = "station"
deviations_pd["entity_id"] = deviations_pd["station_id"]
deviations_pd["entity_name"] = deviations_pd["station_name"]
deviations_pd["magnitude"] = deviations_pd["robust_z"].abs()
deviations_pd = deviations_pd.loc[:, [
    "entity_type", "entity_id", "entity_name", "date", "metric", "metric_group",
    "higher_is_bad", "value", "expected", "deviation", "pct_deviation",
    "robust_z", "magnitude", "direction", "is_outlier",
    "baseline_version", "baseline_phase",
]]

print(f"deviations: {len(deviations_pd):,} rows")
print(f"outliers (|z| >= {OUTLIER_Z}): {int(deviations_pd['is_outlier'].sum()):,}")
display(
    deviations_pd.loc[deviations_pd.is_outlier]
    .sort_values("magnitude", ascending=False)
    .head(12)
)

In [ ]:
# --- 4. write gold tables --------------------------------------------------
base_out = baseline_df.copy()
features_out = features_pd.copy()
dev_out = deviations_pd.copy()

# Spark otherwise infers timestamps; keep a real DATE column for reporting.
features_out["date"] = features_out["date"].dt.date
dev_out["date"] = dev_out["date"].dt.date

write_gold("gold_entity_baseline", spark.createDataFrame(base_out))
write_gold("gold_entity_day_features", spark.createDataFrame(features_out))
write_gold("gold_deviation_daily", spark.createDataFrame(dev_out))

In [ ]:
# --- 5. data-quality record + display -------------------------------------
observed = features_pd["is_observed"]
scored = observed & (features_pd["baseline_phase"] == "scored")
top = deviations_pd.loc[deviations_pd.is_outlier]
if len(top):
    top_row = top.loc[top["magnitude"].idxmax()]
    top_ref = f"{top_row['entity_name']} {top_row['date'].date()} {top_row['metric']} z={top_row['robust_z']:.2f}"
else:
    top_ref = "none"

dq = [
    ("batch_id", BATCH_ID),
    ("baseline_version", BASELINE_VERSION),
    ("estimation_start", str(EST_START.date())),
    ("estimation_end", str(EST_END.date())),
    ("feature_rows", len(features_pd)),
    ("grid_stations", features_pd["station_id"].nunique()),
    ("grid_dates", features_pd["date"].nunique()),
    ("observed_rows", int(observed.sum())),
    ("unobserved_rows", int((~observed).sum())),
    ("grid_completeness_pct", round(float(observed.mean()), 4)),
    ("scored_rows", int(scored.sum())),
    ("baseline_rows", len(baseline_df)),
    ("metrics", len(METRIC_META)),
    ("deviation_rows", len(deviations_pd)),
    ("outlier_rows", int(deviations_pd["is_outlier"].sum())),
    ("max_deviation_score", round(float(features_pd["deviation_score"].max()), 4)),
    ("max_abs_z", round(float(deviations_pd["magnitude"].max()), 4)),
    ("largest_outlier", top_ref),
    ("min_date", str(features_pd["date"].min().date())),
    ("max_date", str(features_pd["date"].max().date())),
]
dq_df = pd.DataFrame(dq, columns=["metric", "value"])
dq_df["value"] = dq_df["value"].astype(str)
write_gold("gold_data_quality", spark.createDataFrame(dq_df))

print("\nGold build complete.")
display(spark.sql("SHOW TABLES").filter(F.col("tableName").startswith("gold_")).orderBy("tableName"))
display(dq_df)

## Next

`nb_04_gold_risk_alerts` consumes `gold_entity_day_features` and
`gold_deviation_daily` to produce:

- a transparent rule-based risk score (`rules_v1`),
- Red / Amber / Green tiers,
- expected barrels and USD at risk,
- `gold_risk_daily`, `gold_alert` (fire-on-change) and `gold_alert_outcome`.

Only rows with `baseline_phase = scored` are eligible for alerts, so the
estimation window can never produce a retrospective warning.
